[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juliopez/Taller-Fundamentos-Data-Science-Python/blob/main/Machine_Learning_2026/03_Notebooks/NB03_Preparacion_Gestion_Dataset_Imagenes.ipynb)
# NB03 — Preparación y gestión del dataset de imágenes
**Correspondencia: Semanas 5–8**

**Dataset de demostración:** TensorFlow Flower Photos (cinco categorías de flores). El mismo procedimiento puede adaptarse al dataset del proyecto integrador.


## 1. Preparación del entorno


In [ ]:
import os
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image

print("TensorFlow:", tf.__version__)


## 2. Fuente de datos y Google Drive (opcional)

Este notebook descarga automáticamente **Flower Photos** desde el repositorio público de ejemplos de TensorFlow. **No se requiere Google Drive** para la demostración. Si la dupla utiliza imágenes propias, puede montar Drive y sustituir `DATASET_DIR` en la sección 3.


In [ ]:
# OPCIONAL: descomente únicamente si su dataset está en Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')
print("Para Flower Photos no es necesario conectar Google Drive.")


## 3. Descargar y ubicar el dataset

Flower Photos se descarga automáticamente y se extrae en una carpeta con una subcarpeta por clase. TensorFlow incluye un archivo de licencia y otros metadatos en el directorio raíz, que no deben contarse como imágenes.

```text
flower_photos/
├── daisy/
├── dandelion/
├── roses/
├── sunflowers/
└── tulips/
```

Para un dataset propio, sustituya la ruta al final de la siguiente celda.


In [ ]:
URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
archivo = tf.keras.utils.get_file("flower_photos.tgz", origin=URL, extract=True)
DATASET_DIR = Path(archivo).parent / "flower_photos"
# Alternativa para dataset propio:
# DATASET_DIR = Path("/content/drive/MyDrive/MachineLearning2026/dataset")
if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f"No se encontró el dataset: {DATASET_DIR}")
print("Dataset:", DATASET_DIR)


## 4. Identificar clases y cantidad de imágenes


In [ ]:
extensiones = {".jpg", ".jpeg", ".png", ".bmp"}
clases = sorted(carpeta.name for carpeta in DATASET_DIR.iterdir() if carpeta.is_dir())
if len(clases) < 2:
    raise ValueError("Se requieren al menos dos carpetas de clases.")
resumen = []
for clase in clases:
    archivos = [f for f in (DATASET_DIR / clase).iterdir() if f.is_file() and f.suffix.lower() in extensiones]
    resumen.append({"Clase": clase, "Cantidad": len(archivos)})
df_resumen = pd.DataFrame(resumen)
print("Total de imágenes:", df_resumen["Cantidad"].sum())
display(df_resumen)


## 5. Analizar el balance de clases


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(df_resumen["Clase"], df_resumen["Cantidad"])
plt.xlabel("Clase")
plt.ylabel("Cantidad de imágenes")
plt.title("Distribución de imágenes por clase")
plt.xticks(rotation=45)
plt.show()


## 6. Visualizar ejemplos del dataset


In [ ]:
fig, axes = plt.subplots(
    len(clases),
    3,
    figsize=(9, max(3, len(clases) * 3))
)

if len(clases) == 1:
    axes = np.array([axes])

for i, clase in enumerate(clases):
    archivos = [
        f for f in (DATASET_DIR / clase).iterdir()
        if f.suffix.lower() in extensiones
    ]

    seleccion = random.sample(
        archivos,
        min(3, len(archivos))
    )

    for j in range(3):
        ax = axes[i, j]
        ax.axis("off")

        if j < len(seleccion):
            imagen = Image.open(seleccion[j]).convert("RGB")
            ax.imshow(imagen)
            ax.set_title(clase)

plt.tight_layout()
plt.show()


## 7. Revisar dimensiones y formatos


In [ ]:
registros = []
for clase in clases:
    for archivo in (DATASET_DIR / clase).iterdir():
        if archivo.suffix.lower() not in extensiones:
            continue
        try:
            with Image.open(archivo) as img:
                registros.append({"archivo": str(archivo), "clase": clase, "ancho": img.width, "alto": img.height, "formato": img.format, "modo": img.mode})
        except Exception as e:
            print("No se pudo leer:", archivo.name, e)
df_imagenes = pd.DataFrame(registros)
display(df_imagenes.head())


In [ ]:
df_imagenes[["ancho", "alto"]].describe()


## 8. Detectar archivos problemáticos

Antes del entrenamiento conviene identificar imágenes dañadas o que no puedan abrirse correctamente.


In [ ]:
archivos_problematicos = []

for clase in clases:
    for archivo in (DATASET_DIR / clase).iterdir():
        if archivo.suffix.lower() not in extensiones:
            continue

        try:
            with Image.open(archivo) as img:
                img.verify()
        except Exception as e:
            archivos_problematicos.append({
                "archivo": str(archivo),
                "error": str(e)
            })

df_problematicos = pd.DataFrame(archivos_problematicos)

print("Archivos problemáticos:", len(df_problematicos))
df_problematicos.head()


## 9. Preparar el dataset con TensorFlow

Se construyen tres subconjuntos **independientes y estratificados**: entrenamiento (70 %), validación (15 %) y test (15 %). La semilla permite reproducir la partición. Primero se separa test y luego validación; el test queda reservado para la evaluación final.

Se excluyen los archivos identificados como problemáticos. Las etiquetas se derivan de los nombres de las carpetas y mantienen un orden alfabético consistente.


In [ ]:
from sklearn.model_selection import train_test_split

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
class_names = clases
clase_a_indice = {nombre: i for i, nombre in enumerate(class_names)}
problematicos = {item["archivo"] for item in archivos_problematicos}
archivos_validos = [(str(p), clase_a_indice[clase]) for clase in clases for p in sorted((DATASET_DIR / clase).iterdir()) if p.is_file() and p.suffix.lower() in extensiones and str(p) not in problematicos]
rutas = [p for p, _ in archivos_validos]
etiquetas = [y for _, y in archivos_validos]
if min(np.bincount(etiquetas)) < 3:
    raise ValueError("Cada clase debe contar con suficientes imágenes para la división estratificada.")
X_train, X_temp, y_train, y_temp = train_test_split(rutas, etiquetas, test_size=0.30, random_state=SEED, stratify=etiquetas)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp)

def cargar_imagen(ruta, etiqueta):
    contenido = tf.io.read_file(ruta)
    imagen = tf.io.decode_image(contenido, channels=3, expand_animations=False)
    imagen.set_shape([None, None, 3])
    imagen = tf.image.resize(imagen, IMG_SIZE)
    return imagen, etiqueta

def crear_dataset(rutas, etiquetas, entrenar=False):
    ds = tf.data.Dataset.from_tensor_slices((rutas, etiquetas))
    if entrenar:
        ds = ds.shuffle(len(rutas), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(cargar_imagen, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = crear_dataset(X_train, y_train, entrenar=True)
val_ds = crear_dataset(X_val, y_val)
test_ds = crear_dataset(X_test, y_test)
print("Clases:", class_names)
print("Train:", len(X_train), "Validation:", len(X_val), "Test:", len(X_test))


## 10. Inspeccionar un batch


In [ ]:
for imagenes, etiquetas in train_ds.take(1):
    print("Forma de las imágenes:", imagenes.shape)
    print("Forma de las etiquetas:", etiquetas.shape)
    print("Rango original:", float(tf.reduce_min(imagenes)), "a", float(tf.reduce_max(imagenes)))


## 11. Normalización

Transformaremos los valores de los píxeles desde el rango 0–255 al rango 0–1.


In [ ]:
normalization_layer = layers.Rescaling(1./255)

train_norm = train_ds.map(
    lambda x, y: (normalization_layer(x), y)
)

val_norm = val_ds.map(
    lambda x, y: (normalization_layer(x), y)
)

for imagenes, etiquetas in train_norm.take(1):
    print(
        "Rango normalizado:",
        float(tf.reduce_min(imagenes)),
        "a",
        float(tf.reduce_max(imagenes))
    )


## 12. División Train / Validation / Test

La división **70/15/15**, estratificada y reproducible, ya se realizó en la sección 9. El conjunto de entrenamiento permite ajustar parámetros; validación sirve para seleccionar y ajustar el modelo; **test no debe consultarse para tomar decisiones de entrenamiento**.

Los objetos `train_ds`, `val_ds` y `test_ds` están disponibles en memoria. Para un proyecto propio, documente la proporción utilizada, la distribución por clase y cualquier criterio especial de separación (por ejemplo, evitar imágenes duplicadas o del mismo sujeto entre subconjuntos).


## 13. Data Augmentation

El aumento de datos permite generar variaciones de las imágenes de entrenamiento sin modificar las etiquetas.


In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10)
])

for imagenes, etiquetas in train_ds.take(1):
    imagen = imagenes[0]

    plt.figure(figsize=(8, 8))

    for i in range(9):
        imagen_aumentada = data_augmentation(
            tf.expand_dims(imagen, 0),
            training=True
        )

        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(
            tf.cast(imagen_aumentada[0], tf.uint8)
        )
        plt.axis("off")

    plt.tight_layout()
    plt.show()


## 14. Incorporar Data Augmentation al pipeline


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
# Solo el entrenamiento recibe transformaciones aleatorias.
# La normalización 0–1 se aplica a los tres subconjuntos.
train_aug = train_ds.map(lambda x, y: (normalization_layer(data_augmentation(x, training=True)), y), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_preparado = val_ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_preparado = test_ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
print("Pipelines listos: train_aug, val_preparado, test_preparado")


## 15. Verificar etiquetas y clases

El orden de las clases será importante posteriormente, porque deberá mantenerse durante el entrenamiento, la exportación del modelo y la aplicación final.


In [ ]:
for indice, clase in enumerate(class_names):
    print(indice, "→", clase)


## 16. Registrar las características del dataset


In [ ]:
ficha_dataset = {
    "fuente": "TensorFlow Flower Photos",
    "numero_imagenes_validas": len(archivos_validos),
    "numero_clases": len(class_names),
    "clases": class_names,
    "distribucion_por_clase": df_resumen.set_index("Clase")["Cantidad"].to_dict(),
    "tamano_imagen": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "particion": {"train": len(X_train), "validation": len(X_val), "test": len(X_test)},
    "normalizacion": "0-1",
    "data_augmentation_solo_train": ["RandomFlip horizontal", "RandomRotation 0.10", "RandomZoom 0.10"],
    "archivos_problematicos": len(archivos_problematicos)
}
ficha_dataset


## 17. Actividad

Aplique el notebook al dataset seleccionado para el proyecto integrador y registre:

1. Número total de imágenes.
2. Número de clases.
3. Cantidad de imágenes por clase.
4. Ejemplos representativos de cada categoría.
5. Dimensiones originales de las imágenes.
6. Archivos problemáticos detectados.
7. Estrategia de redimensionamiento y normalización.
8. Distribución definida para train, validation y test.
9. Existencia de desbalance entre clases.
10. Transformaciones de Data Augmentation seleccionadas y su justificación.


## 18. Base para la Evaluación 2

El resultado debe ser un **dataset preparado, organizado y validado**, listo para la futura construcción y entrenamiento de una CNN.

**Dataset original → exploración → control de calidad → etiquetado → división estratificada → redimensionamiento → normalización → Data Augmentation solo en train → dataset preparado**.

Los pipelines finales son `train_aug`, `val_preparado` y `test_preparado`. No se entrena una CNN en este notebook.
